# DeltaNet State Prediction Residual

Replay DeltaNet layer states and plot the pre-update residual `||S_{t-1} k_t - v_t||` over sequence length. This is the term DeltaNet multiplies by `beta_t` before writing into the state, so a falling residual would indicate an implicit learning-rate decay from better state predictions.

In [ ]:
from __future__ import annotations

import gc
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / "PFNs").exists():
    repo_root = repo_root.parent
if not (repo_root / "PFNs").exists():
    raise RuntimeError("Could not find repo root containing PFNs/.")
sys.path.insert(0, str(repo_root / "PFNs"))

from fla.layers.delta_net import DeltaNet
from pfns.experiments.model_benchmarks.model_registry import MODEL_FAMILIES
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark
from pfns.model.fla_patches import _apply_deltanet_beta_decay

warnings.filterwarnings("ignore", message="ShortConvolution is crucial.*", category=UserWarning)
plt.rcParams["figure.dpi"] = 120

## Configuration

In [ ]:
MODEL_NAMES = [
    "Non_Causal_DeltaNet",
    "Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform",
    "DeltaNet_Comb_ST_Reference",
]
MODEL_LABELS = {
    "Non_Causal_DeltaNet": "NC DeltaNet",
    "Non_Causal_DeltaNet_with_lr_decay_online_inverse_t0_256": "NC online inverse",
    "DeltaNet_Comb_ST_Reference": "Reference",
    "DeltaNet_Comb_ST_Seq_Len_200-64K_loguniform": "200-64K",
}
LAYER_INDICES = [0, 6, 11]
TRAIN_CONTEXT_LEN = 128_000
N_TEST = 64
BATCH_SIZE = 1
NUM_FEATURES = 20
MAX_POINTS = 4096
POSITION_SAMPLING = "log"  # "log" or "linear"
SMOOTH_WINDOW = 31
EPS = 1e-30

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AUTOCAST_DTYPE = torch.bfloat16 if DEVICE.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float16
USE_AUTOCAST = DEVICE.type == "cuda"
print(f"device={DEVICE}, use_autocast={USE_AUTOCAST}, autocast_dtype={AUTOCAST_DTYPE}")

## Helpers

In [ ]:
def label(model_name: str) -> str:
    return MODEL_LABELS.get(model_name, model_name)


def find_config(model_name: str) -> dict:
    for registry in MODEL_FAMILIES.values():
        if model_name in registry:
            return registry[model_name]
    raise KeyError(f"Unknown model: {model_name}")


def load_model(model_name: str):
    models, configs = load_models_for_benchmark({model_name: model_configs[model_name]}, device=str(DEVICE))
    models[model_name].eval()
    return models[model_name], configs[model_name]


def deltanet_layers(model):
    return [(name, layer) for name, layer in model.named_modules() if isinstance(layer, DeltaNet)]


def sample_positions(seq_len: int, max_points: int = MAX_POINTS) -> set[int]:
    if POSITION_SAMPLING == "linear":
        points = np.linspace(0, seq_len - 1, min(seq_len, max_points)).round().astype(int)
    elif POSITION_SAMPLING == "log":
        points = np.r_[
            np.arange(min(seq_len, 64, max_points), dtype=int),
            np.geomspace(1, seq_len, max_points * 4).round().astype(int) - 1,
        ]
    else:
        raise ValueError(f"Unknown POSITION_SAMPLING={POSITION_SAMPLING!r}")

    required = {0, min(999, seq_len - 1), seq_len - 1}
    points = sorted({int(p) for p in points if 0 <= p < seq_len} | required)
    if len(points) <= max_points:
        return set(points)

    keep = set(points[:64]) | required
    rest = [p for p in points if p not in keep]
    need = max_points - len(keep)
    rest = [rest[i] for i in np.linspace(0, len(rest) - 1, need).round().astype(int)] if need > 0 else []
    return keep | set(rest)


def collect_delta_inputs(model, model_name: str, layer_index: int, x: torch.Tensor, y: torch.Tensor) -> dict:
    layer_name, layer = deltanet_layers(model)[layer_index]
    assert not layer.use_short_conv, "This notebook assumes use_short_conv=False."
    assert not layer.allow_neg_eigval, "This notebook assumes allow_neg_eigval=False."
    assert layer.qk_norm == "l2", f"This notebook assumes qk_norm='l2', got {layer.qk_norm!r}."
    assert layer.qk_activation in {"silu", "relu", "elu", "identity"}
    captured = {}

    def hook(module, args, kwargs):
        hidden = kwargs["hidden_states"] if "hidden_states" in kwargs else args[0]
        batch, seq_len = hidden.shape[:2]

        k = module.k_proj(hidden)
        if module.qk_activation == "silu":
            k = F.silu(k)
        elif module.qk_activation == "relu":
            k = torch.relu(k)
        elif module.qk_activation == "elu":
            k = F.elu(k) + 1.0

        v = F.silu(module.v_proj(hidden))
        beta = module.b_proj(hidden).sigmoid() if module.use_beta else hidden.new_ones(batch, seq_len, module.num_heads)
        beta = _apply_deltanet_beta_decay(
            beta,
            mode=getattr(getattr(model, "transformer_layers", None), "deltanet_beta_decay", "none"),
            t0=int(getattr(getattr(model, "transformer_layers", None), "deltanet_beta_decay_t0", 1000)),
            position_dim=1,
        )

        k = k.view(batch, seq_len, module.num_heads, module.head_k_dim).movedim(1, 0)
        v = v.view(batch, seq_len, module.num_heads, module.head_v_dim).movedim(1, 0)
        k = F.normalize(k, p=2, dim=-1, eps=1e-6)

        captured["k"] = k.reshape(seq_len, -1, module.head_k_dim).detach().float().cpu()
        captured["v"] = v.reshape(seq_len, -1, module.head_v_dim).detach().float().cpu()
        captured["beta"] = beta.movedim(1, 0).reshape(seq_len, -1).detach().float().cpu()

    handle = layer.register_forward_pre_hook(hook, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type=DEVICE.type, dtype=AUTOCAST_DTYPE, enabled=USE_AUTOCAST):
            _ = model.incontext_fit(x, y)
    finally:
        handle.remove()

    assert {"k", "v", "beta"} <= captured.keys(), "DeltaNet hook did not run."
    return {"model_name": model_name, "model_label": label(model_name), "layer_index": layer_index, "layer": layer_name, **captured}


def residual_curve(record: dict) -> pd.DataFrame:
    k = record["k"].to(DEVICE)
    v = record["v"].to(DEVICE)
    beta = record["beta"].to(DEVICE)
    seq_len, obs, key_dim = k.shape
    state = torch.zeros(obs, key_dim, v.shape[-1], device=DEVICE)
    positions = sample_positions(seq_len)
    rows = []

    for t in range(seq_len):
        prediction = torch.einsum("ok,okv->ov", k[t], state)
        residual = v[t] - prediction

        if t in positions:
            residual_norm = residual.norm(dim=-1).clamp_min(EPS)
            value_norm = v[t].norm(dim=-1).clamp_min(EPS)
            rows.append({
                "model_name": record["model_name"],
                "model_label": record["model_label"],
                "layer_index": record["layer_index"],
                "position": t,
                "mean_residual": float(residual_norm.mean()),
                "relative_residual": float((residual_norm / value_norm).mean()),
                "update_norm": float((beta[t].abs() * residual_norm).mean()),
                "mean_beta": float(beta[t].mean()),
                "pred_value_cosine": float(F.cosine_similarity(prediction, v[t], dim=-1, eps=1e-12).mean()),
            })

        state += beta[t].view(obs, 1, 1) * k[t].unsqueeze(-1) * residual.unsqueeze(-2)

    del k, v, beta, state
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return pd.DataFrame(rows)


## Run Diagnostics

In [ ]:
model_configs = {name: find_config(name) for name in MODEL_NAMES}
base_model, base_config = load_model(MODEL_NAMES[0])
batch = base_config.priors[0].create_get_batch_method()(
    batch_size=BATCH_SIZE,
    seq_len=TRAIN_CONTEXT_LEN + N_TEST,
    num_features=NUM_FEATURES,
    single_eval_pos=TRAIN_CONTEXT_LEN,
    n_targets_per_input=1,
)
train_x = batch.x[:, :TRAIN_CONTEXT_LEN].to(DEVICE)
train_y = batch.y[:, :TRAIN_CONTEXT_LEN].to(DEVICE)
print(f"train_x={tuple(train_x.shape)}, train_y={tuple(train_y.shape)}")

curves = []
for i, model_name in enumerate(MODEL_NAMES):
    model = base_model if i == 0 else load_model(model_name)[0]
    try:
        for layer_index in LAYER_INDICES:
            record = collect_delta_inputs(model, model_name, layer_index, train_x, train_y)
            print(
                f"{label(model_name)} layer {layer_index}: "
                f"k={tuple(record['k'].shape)}, v={tuple(record['v'].shape)}, beta={tuple(record['beta'].shape)}"
            )
            curves.append(residual_curve(record))
            del record
            gc.collect()
    finally:
        if i != 0:
            del model
            gc.collect()
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

residual_df = pd.concat(curves, ignore_index=True)
display(residual_df.head())


## Plots

In [ ]:
COLORS = {name: f"C{i}" for i, name in enumerate(MODEL_NAMES)}
METRICS = {
    "mean_residual": "mean ||S_{t-1} k_t - v_t||",
    "relative_residual": "mean residual / ||v_t||",
    "update_norm": "mean beta_t * residual",
}


def smooth(y: np.ndarray, window: int = SMOOTH_WINDOW) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if window <= 1 or y.size < 3:
        return y
    window = min(int(window) | 1, y.size if y.size % 2 else y.size - 1)
    return np.convolve(np.pad(y, window // 2, mode="edge"), np.ones(window) / window, mode="valid")


def plot_metric(metric: str) -> None:
    for layer_index in LAYER_INDICES:
        fig, ax = plt.subplots(figsize=(10, 3.5))
        for model_name in MODEL_NAMES:
            curve = residual_df[(residual_df.model_name == model_name) & (residual_df.layer_index == layer_index)].sort_values("position")
            ax.plot(curve.position + 1, smooth(curve[metric]), label=label(model_name), color=COLORS[model_name], linewidth=1.8)
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel(f"sequence position ({POSITION_SAMPLING}-sampled)")
        ax.set_ylabel(METRICS[metric])
        ax.set_title(f"{METRICS[metric]} | layer {layer_index}")
        ax.grid(True, which="both", alpha=0.25)
        ax.legend(fontsize=8)
        fig.tight_layout()
        plt.show()


for metric in METRICS:
    plot_metric(metric)

## Summary

In [ ]:
def nearest_rows(position: int) -> pd.DataFrame:
    return pd.DataFrame([curve.loc[(curve.position - position).abs().idxmin()] for _, curve in residual_df.groupby(["model_name", "layer_index"])])

near_1k = nearest_rows(1_000).set_index(["model_name", "layer_index"])
near_end = nearest_rows(TRAIN_CONTEXT_LEN - 1).set_index(["model_name", "layer_index"])
summary = near_end[["model_label", "position", "mean_residual", "relative_residual", "update_norm", "mean_beta"]].rename(columns={"position": "final_position"})
summary["position_near_1k"] = near_1k.position
summary["residual_near_1k"] = near_1k.mean_residual
summary["final_over_1k_residual"] = summary.mean_residual / summary.residual_near_1k.clip(lower=EPS)
display(summary.reset_index().sort_values(["layer_index", "model_name"]))